## Goal

The objective of this sprint is to implement a Support Vector Machine (SVM) classifier and compare its performance with previously developed baseline models. This sprint focuses on understanding margin maximization, decision boundaries, and evaluating whether SVM provides better generalization for customer churn prediction.

## Research Notes

1) Why do we evaluate Support Vector Machine (SVM) after Random Forest?
2) What is the main objective of an SVM classifier?
3) Why is feature scaling particularly important for SVM?
4) What is the role of the kernel function in SVM?
5) If an SVM model achieves higher Accuracy but lower Recall than Logistic Regression, which model would you choose for customer churn prediction? Why?
6) Is Support Vector Machine always a better choice than Logistic Regression?

---

- 1-SVM is evaluated after Random Forest because it introduces a fundamentally different learning strategy. Unlike tree-based models, SVM focuses on finding the optimal decision boundary with the maximum margin between classes. Furthermore, kernel functions enable SVM to model complex non-linear relationships.

- 2-The primary objective of an SVM classifier is to identify the optimal decision boundary that maximizes the margin between different classes. A larger margin generally improves the model's ability to generalize to unseen data.

- 3-Feature scaling is important for SVM because it is a distance-based algorithm. Features with larger scales can dominate the decision boundary. Therefore, using StandardScaler is generally necessary to put all features on a similar scale and improve model performance.

- 4-Many real-world datasets cannot be separated using a linear decision boundary. Kernel functions enable SVM to model non-linear decision boundaries by implicitly mapping the data into a higher-dimensional feature space, where linear separation becomes possible.

- 5-For customer churn prediction, Recall is generally more important than Accuracy because identifying customers who are likely to leave has greater business value. Therefore, a model with slightly lower Accuracy but substantially higher Recall may be the preferred choice.

- 6-No. Model selection should consider multiple factors beyond predictive performance, including computational cost, training time, inference speed, scalability, model complexity, and generalization capability. Most importantly, the selected model should satisfy the business requirements of the problem rather than maximizing a single evaluation metric.

In [8]:
# Imports 
import pandas as pd 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [9]:
# Load Clean Dataset
df = pd.read_csv('../data/processed/customer_churn_clean.csv')
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [10]:
# Feature & Target Separation
X = df.drop(columns='Churn')
y = df['Churn'].map({"No": 0, "Yes": 1})

In [11]:
# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y , test_size=0.3, random_state=42, stratify=y)

In [12]:
# Feature Lists
binary_features = ['gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling']

multiclass_features = ['MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaymentMethod']

numerical_features = ['tenure', 'MonthlyCharges', 'TotalCharges']

passthrough_features = [
    "SeniorCitizen"
]

In [13]:
# ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        (
            'binary',
            OrdinalEncoder(),
            binary_features
        ),
        (
            'multiclass',
            OneHotEncoder(handle_unknown='ignore'),
            multiclass_features
        ),
        (
            'numerical',
            StandardScaler(),
            numerical_features
        ),
        (
            'pass',
            'passthrough',
            passthrough_features
        )
    ]
)

In [14]:
# Previous Baseline Pipelines
baseline_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('baseline_model', LogisticRegression(random_state=42))
])
dt_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('default_dt', DecisionTreeClassifier(random_state=42))
])
knn_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('knn', KNeighborsClassifier())
])
nb_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('gaussian_nb', GaussianNB())
])
rf_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('rf', RandomForestClassifier(random_state=42))
])

In [15]:
# Previous baseline model fit
baseline_pipe.fit(X_train, y_train)
dt_pipe.fit(X_train, y_train)
knn_pipe.fit(X_train, y_train)
nb_pipe.fit(X_train, y_train)
rf_pipe.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('rf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](19,)","['gender','SeniorCitizen','Partner',...,'PaymentMethod','MonthlyCharges', 'TotalCharges']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,19
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('binary', ...), ('multiclass', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``r

In [16]:
# SVC Pipeline
svc_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('svc', SVC())
])

In [17]:
# Model Training
svc_pipe.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('svc', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](19,)","['gender','SeniorCitizen','Partner',...,'PaymentMethod','MonthlyCharges', 'TotalCharges']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,19
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('binary', ...), ('multiclass', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``

In [18]:
# Prediction
baseline_pred = baseline_pipe.predict(X_test)
dt_pred = dt_pipe.predict(X_test)
knn_pred = knn_pipe.predict(X_test)
nb_pred = nb_pipe.predict(X_test)
rf_pred = rf_pipe.predict(X_test)
svc_pred = svc_pipe.predict(X_test)

In [19]:
# Model Evaluation
preds = [baseline_pred, dt_pred, knn_pred, nb_pred, rf_pred, svc_pred]
model_names = ['LogisticRegression', 'DecisionTreeClassifier', 'KNN', 'GaussianNB', 'RandomForest(Default)', 'SVC']
metrics_list = []
cm_list = []
cr_list = []
def metrics(y_true, y_pred):
    return {
        'Accuracy': accuracy_score(y_true, y_pred),
        'Precision': precision_score(y_true, y_pred), 
        'Recall': recall_score(y_true, y_pred), 
        'F1': f1_score(y_true, y_pred), 
    }
for pred in preds:
    metrics_list.append(metrics(y_test, pred))
    cm_list.append(confusion_matrix(y_test, pred))
    cr_list.append(classification_report(y_test, pred))

In [20]:
for model, metric in zip(model_names, metrics_list):
    print(f'{model} Metrics:\n {metric}')
    print(100*'=')

LogisticRegression Metrics:
 {'Accuracy': 0.8056872037914692, 'Precision': 0.6556701030927835, 'Recall': 0.5668449197860963, 'F1': 0.6080305927342257}
DecisionTreeClassifier Metrics:
 {'Accuracy': 0.7118483412322275, 'Precision': 0.45811051693404636, 'Recall': 0.45811051693404636, 'F1': 0.45811051693404636}
KNN Metrics:
 {'Accuracy': 0.7611374407582938, 'Precision': 0.5515370705244123, 'Recall': 0.5436720142602496, 'F1': 0.547576301615799}
GaussianNB Metrics:
 {'Accuracy': 0.6791469194312796, 'Precision': 0.4453860640301318, 'Recall': 0.8431372549019608, 'F1': 0.5828712261244609}
RandomForest(Default) Metrics:
 {'Accuracy': 0.7796208530805687, 'Precision': 0.6121495327102804, 'Recall': 0.46702317290552586, 'F1': 0.5298281092012134}
SVC Metrics:
 {'Accuracy': 0.795260663507109, 'Precision': 0.655421686746988, 'Recall': 0.48484848484848486, 'F1': 0.5573770491803278}


In [21]:
# Confusion Matrix
for model, cm in zip(model_names, cm_list):
    print(f'{model} Confusion Matrix:\n {cm}')
    print(60*'=')

LogisticRegression Confusion Matrix:
 [[1382  167]
 [ 243  318]]
DecisionTreeClassifier Confusion Matrix:
 [[1245  304]
 [ 304  257]]
KNN Confusion Matrix:
 [[1301  248]
 [ 256  305]]
GaussianNB Confusion Matrix:
 [[960 589]
 [ 88 473]]
RandomForest(Default) Confusion Matrix:
 [[1383  166]
 [ 299  262]]
SVC Confusion Matrix:
 [[1406  143]
 [ 289  272]]


In [22]:
# Classification Report
for model, cr in zip(model_names, cr_list):
    print(f'{model} Classification Report:\n {cr}')
    print(60*'=')

LogisticRegression Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.89      0.87      1549
           1       0.66      0.57      0.61       561

    accuracy                           0.81      2110
   macro avg       0.75      0.73      0.74      2110
weighted avg       0.80      0.81      0.80      2110

DecisionTreeClassifier Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.80      0.80      1549
           1       0.46      0.46      0.46       561

    accuracy                           0.71      2110
   macro avg       0.63      0.63      0.63      2110
weighted avg       0.71      0.71      0.71      2110

KNN Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.84      0.84      1549
           1       0.55      0.54      0.55       561

    accuracy                           0.76      2110
   macro a

In [23]:
# Comparison with Previous Models
comparison_df = pd.DataFrame(metrics_list, index=model_names)
comparison_df

,Accuracy,Precision,Recall,F1
LogisticRegression,0.805687,0.655670,0.566845,0.608031
DecisionTreeClassifier,0.711848,0.458111,0.458111,0.458111
KNN,0.761137,0.551537,0.543672,0.547576
GaussianNB,0.679147,0.445386,0.843137,0.582871
RandomForest(Default),0.779621,0.612150,0.467023,0.529828
SVC,0.795261,0.655422,0.484848,0.557377


## Key Findings

- Support Vector Machine achieved competitive performance compared to Logistic Regression.
- SVM produced nearly identical Precision but lower Recall than Logistic Regression.
- The default SVM adopted a more conservative decision boundary, reducing False Positives while increasing False Negatives.
- Logistic Regression remained the strongest baseline model for customer churn prediction.
- Hyperparameter tuning may further improve SVM performance in future sprints.

## Sprint Retrospective

### What went well

- Successfully implemented the baseline Support Vector Machine pipeline.
- Evaluated SVM using the same preprocessing and evaluation framework as previous models.
- Maintained a fair comparison across all baseline classifiers.

### Challenges

- Although SVM achieved competitive Accuracy, its Recall was lower than Logistic Regression.
- Interpreting the trade-off between Precision and Recall required careful analysis.

### Lessons Learned

- SVM maximizes the decision margin rather than simply separating classes.
- Feature scaling is essential for distance-based optimization algorithms like SVM.
- A more sophisticated algorithm does not necessarily outperform a simpler linear model.
- Business requirements should guide the final model selection.

### Next Steps

- Build a complete comparison of all baseline models.
- Begin hyperparameter tuning for the most promising classifiers.
- Investigate techniques to improve Recall while maintaining acceptable Precision.

In [ ]:
# 